# Vežba 1: ML proces i deskriptivna analiza

Faze projekta mašinskog učenja, Google Colab i prvi pogled na podatke o saobraćajnim nezgodama u Srbiji

Lekcija sa objašnjenjima: [https://isatvss.surge.sh/vezba1.html](https://isatvss.surge.sh/vezba1.html)

Ćelije se izvršavaju redom, od vrha nadole (Shift+Enter), jer svaka koristi promenljive iz prethodnih.

## Učitavanje i struktura podataka

### Zadatak 1. Učitavanje skupa podataka

Učitati CSV fajl sa adrese praktikuma, pa ispisati dimenzije tabele, prvi zapis i prvih pet redova za tri izabrane kolone.

In [ ]:
import pandas as pd

URL = "https://isatvss.surge.sh/podaci/nezgode_2025.csv"
nezgode = pd.read_csv(URL)

print("Broj redova i kolona:", nezgode.shape)
print("\nPrvi zapis:")
print(nezgode.iloc[0])
print("\nPrvih pet redova, izabrane kolone:")
print(nezgode[["policijska_uprava", "datum_vreme", "posledica"]].head())

### Zadatak 2. Tipovi kolona, nedostajuće vrednosti i ponovljeni zapisi

Proveriti tip svake kolone i broj nedostajućih vrednosti, a zatim i da li se neki identifikator nezgode pojavljuje više puta.

Identifikator se ponavlja 228 puta, ali potpuno istih redova ima samo 2. Ponovljen identifikator, dakle, najčešće nije greška pri unosu. Ista nezgoda upisana je u više redova jer joj je dodeljeno više vrsta ili opisa, kao nezgodi 1508025 u ispisu. Kako takve redove obraditi, odlučuje se na vežbi 2. Ovde je važno nešto drugo: pretpostavka o podacima, na primer „ponovljen identifikator je duplikat", proverava se kodom pre nego što se na njoj zasnuje odluka.

In [ ]:
nezgode.info()

print("\nNedostajuće vrednosti po koloni:")
print(nezgode.isna().sum())

print("\nBroj ponovljenih id_nezgode:", nezgode["id_nezgode"].duplicated().sum())
print("Broj potpuno istih redova:", nezgode.duplicated().sum())

ponovljeni = nezgode[nezgode["id_nezgode"] == 1508025]
print(ponovljeni[["id_nezgode", "datum_vreme", "vrsta"]].to_string())

## Ciljna promenljiva i osnovni model

### Zadatak 3. Raspodela klasa

Izbrojati nezgode po posledici, u apsolutnom broju i u procentima.

In [ ]:
print(nezgode["posledica"].value_counts())

udeli = nezgode["posledica"].value_counts(normalize=True) * 100
print(udeli.round(1))

### Zadatak 4. Binarna ciljna promenljiva i osnovni model

Spojiti nezgode sa povređenim i poginulim u jednu klasu, teška nezgoda (1), nasuprot nezgodi sa materijalnom štetom (0). Zatim izračunati udeo teških nezgoda i tačnost modela koji za svaku nezgodu predviđa najčešću klasu.

Model koji uopšte ne gleda podatke i uvek procenjuje materijalnu štetu ima tačnost 59,3 %. To je **osnovni model** (baseline). Svaki pravi model mora da ga nadmaši, a u izveštaju se rezultat modela uvek navodi uz rezultat osnovnog. Tačnost od šezdeset procenata zvuči pristojno, ali na ovim podacima ne bi značila gotovo ništa.

In [ ]:
nezgode["teska"] = (nezgode["posledica"] != "Sa mat.stetom").astype(int)

udeo_teskih = nezgode["teska"].mean()
print(f"Udeo nezgoda sa povređenim ili poginulim: {udeo_teskih:.1%}")
print(f"Tačnost modela koji uvek predviđa materijalnu štetu: {1 - udeo_teskih:.1%}")

## Izvedena obeležja iz vremena

### Zadatak 5. Sat, dan u nedelji i mesec

Pretvoriti kolonu `datum_vreme` u datum, izvesti tri nove kolone i ispisati njihovu deskriptivnu statistiku.

In [ ]:
vreme = pd.to_datetime(nezgode["datum_vreme"], format="%d.%m.%Y,%H:%M")

nezgode["sat"] = vreme.dt.hour
nezgode["dan_u_nedelji"] = vreme.dt.dayofweek
nezgode["mesec"] = vreme.dt.month

print(nezgode[["sat", "dan_u_nedelji", "mesec"]].describe().round(2))

### Zadatak 6. Zašto se format uvek navodi

Pročitati istu kolonu bez parametra `format` i uporediti rezultat sa ispravnim datumima iz prethodnog zadatka.

Pogrešno je pročitano 11.995 od 33.415 datuma, svaki u kome je dan manji ili jednak 12, a različit od meseca. Sva obeležja izvedena iz takvih datuma bila bi pogrešna, a model bi se uredno trenirao i na njima. Upozorenja u Colab-u se zato čitaju, a ne preskaču.

In [ ]:
pogodjeno = pd.to_datetime(nezgode["datum_vreme"])

poredjenje = pd.DataFrame({"tekst": nezgode["datum_vreme"], "bez_formata": pogodjeno, "sa_formatom": vreme})
print(poredjenje.head(3).to_string())
print("\nMesec bez formata, od", pogodjeno.dt.month.min(), "do", pogodjeno.dt.month.max())
print("Pogrešno pročitanih datuma:", (pogodjeno != vreme).sum())

## Deskriptivna statistika kategorija

### Zadatak 7. Vrste nezgoda i policijske uprave

Izbrojati nezgode po vrsti i po policijskoj upravi, kao i broj različitih detaljnih opisa.

In [ ]:
print(nezgode["vrsta"].value_counts())
print()
print(nezgode["policijska_uprava"].value_counts().head(8))
print("\nRazličitih opisa nezgode:", nezgode["opis"].nunique())

### Zadatak 8. Udeo teških nezgoda po vrsti

Za svaku vrstu nezgode izračunati broj nezgoda i procenat teških, pa tabelu poređati po procentu.

Vrsta nezgode snažno razdvaja klase. Kod nezgoda sa pešacima 98,4 % je teških, a kod nezgoda sa parkiranim vozilima 7,3 %. Ovo obeležje biće među najkorisnijim za model, ali uz pitanje koje se postavlja u svakom projektu: da li je vrednost obeležja poznata u trenutku kada se procena pravi? Ovde jeste, jer se vrsta nezgode utvrđuje na licu mesta. Obeležje koje nastaje tek posle ishoda dalo bi modelu odgovor unapred, što se naziva curenje podataka i obrađuje na vežbi 2.

In [ ]:
po_vrsti = nezgode.groupby("vrsta")["teska"].agg(broj="count", udeo_teskih="mean")
po_vrsti["udeo_teskih"] = (po_vrsti["udeo_teskih"] * 100).round(1)

print(po_vrsti.sort_values("udeo_teskih", ascending=False))

## Vizuelizacija

### Zadatak 9. Nezgode po satu u danu

Nacrtati dva grafika jedan pored drugog: broj nezgoda po satu i procenat teških nezgoda po satu, sa linijom proseka. Ispisati i vrednosti za četiri izabrana sata.

Grafik pokazuje nešto što se nije moglo unapred pretpostaviti. Noću ima manje nezgoda, ali je i udeo teških manji: 29,3 % u 3 časa, naspram 42,6 % u 15 časova. Pre tumačenja treba pogledati broj slučajeva. U 3 časa zabeležene su 434 nezgode, a u 15 časova 2.190, pa je noćna procena manje pouzdana. Moguće objašnjenje je drugačiji sastav nezgoda noću, na primer više nezgoda sa parkiranim vozilima. To je pretpostavka koju tek treba proveriti, a ne zaključak.

In [ ]:
import matplotlib.pyplot as plt

po_satu = nezgode.groupby("sat")["teska"].agg(broj="count", udeo_teskih="mean")

fig, (levo, desno) = plt.subplots(1, 2, figsize=(11, 4))
levo.bar(po_satu.index, po_satu["broj"], color="#5b3cc4")
levo.set_title("Broj nezgoda po satu")
levo.set_xlabel("Sat u danu")
levo.set_ylabel("Broj nezgoda")

desno.plot(po_satu.index, po_satu["udeo_teskih"] * 100, marker="o", color="#d9480f")
desno.axhline(udeo_teskih * 100, linestyle="--", color="gray", label="prosek")
desno.set_title("Udeo teških nezgoda po satu")
desno.set_xlabel("Sat u danu")
desno.set_ylabel("Udeo teških (%)")
desno.legend()

plt.tight_layout()
plt.show()

izabrani_sati = po_satu.loc[[3, 8, 15, 22]].copy()
izabrani_sati["udeo_teskih"] = (izabrani_sati["udeo_teskih"] * 100).round(1)
print(izabrani_sati)

### Zadatak 10. Nezgode na području Niša

Izdvojiti nezgode sa područja Policijske uprave Niš, uporediti udeo teških sa celom Srbijom, izbrojati nezgode po opštini i nacrtati nezgode na teritoriji grada Niša prema geografskim koordinatama.

Na području uprave zabeležena je 1.321 nezgoda, a udeo teških je 47,5 %, naspram 40,7 % u celoj Srbiji. Razlika može poticati od drugačijeg sastava nezgoda, pa se bez dodatne provere ne tumači kao veća opasnost. Na mapi same tačke iscrtavaju putnu mrežu, sa gustim jezgrom grada i izlaznim pravcima.

In [ ]:
nis = nezgode[nezgode["policijska_uprava"] == "NIŠ"]

print("Nezgoda na području PU Niš:", len(nis))
print(f"Udeo teških u PU Niš: {nis['teska'].mean():.1%}")
print(nis["opstina"].value_counts())

grad = nis[nis["opstina"].isin(["NIŠ", "NIŠKA BANJA"])]
plt.figure(figsize=(6, 6))
for teska, boja, oznaka in [(0, "#9aa5b1", "materijalna šteta"), (1, "#d9480f", "povređeni ili poginuli")]:
    deo = grad[grad["teska"] == teska]
    plt.scatter(deo["lon"], deo["lat"], s=6, alpha=0.5, color=boja, label=oznaka)
plt.title("Nezgode na teritoriji grada Niša, 2025.")
plt.xlabel("Geografska dužina")
plt.ylabel("Geografska širina")
plt.legend()
plt.show()

## Zadaci za samostalan rad

### Zadatak 11. Pregled skupa o kvalitetu vazduha

Učitati `https://isatvss.surge.sh/podaci/vazduh_nis.csv` tako da kolona `datum` bude tipa datum (parametar `parse_dates`). Ispisati period merenja, broj nedostajućih vrednosti po koloni i deskriptivnu statistiku za PM10, PM2.5, temperaturu i vetar.

### Zadatak 12. Dani sa prekoračenjem granične vrednosti

Granična vrednost dnevne koncentracije PM10 je 50 µg/m³ i sme biti prekoračena najviše 35 dana u kalendarskoj godini, prema Uredbi o uslovima za monitoring i zahtevima kvaliteta vazduha (Službeni glasnik RS 11/2010, 75/2010 i 63/2013). Za svaku godinu izbrojati dane sa koncentracijom iznad 50 i dane u kojima merenje postoji. Da li se broj prekoračenja može direktno porediti između godina?

### Zadatak 13. Sezonski obrazac

Izračunati prosečnu koncentraciju PM10 po mesecu i nacrtati stubičasti grafik sa vodoravnom linijom granične vrednosti. U kojim mesecima je prosek iznad granice i čime bi se to moglo objasniti?

### Zadatak 14. Veza PM10 sa ostalim veličinama

Izračunati koeficijent korelacije PM10 sa PM2.5, NO2 i meteorološkim veličinama. Koja meteorološka veličina je najjače povezana sa PM10 i kakvog je smera ta veza?

### Zadatak 15. Teške nezgode po danu u nedelji

Na skupu o nezgodama izračunati broj nezgoda i procenat teških po danu u nedelji, sa nazivima dana umesto brojeva. Da li se vikend razlikuje od radnih dana?